Raw and messy (but informative) notebook for quick data analysis.

Note: we have anonymised this repo so this notebook might not work very well. We will release the fully working version upon acceptance

# Baseline

Quick and dirty predictions; we'll redo these in the cells below

In [ ]:
data = [json.loads(l) for l in (open(f"finetuned_predictions/human/Geordie/dev-qwen-3-8b_0_per_crit.json", "r", encoding='utf-8')).readlines()][-200:]
ixes = [p["Index"] for p in data]

for m in ["dev-qwen-3-8b", "dev-anthropic-claude-opus-4-1", "dev-gpt-5-mini",
          "dev-phi-4", "dev-gpt-41-longco-2025-04-14"]:
    data = [json.loads(l) for l in (open(f"icl_predictions/Geordie/{m}_0_all_crit.json", "r", encoding='utf-8')).readlines()][-200:]
    data = [p for p in data if p["Index"] in ixes]

    for s in ['NoBreakdown', 'BreakdownNoReasons', 'BreakdownReasons']:
        print("\t", s, compute_acc(data, s))

print("\n========")
print("Finetuned")
print("========\n")


for m in ["dev-qwen-3-8b", "dev-anthropic-claude-opus-4-1", "dev-gpt-5-mini",
          "dev-phi-4", "dev-gpt-41-longco-2025-04-14"]:
    data = [json.loads(l) for l in (open(f"icl_predictions/Geordie/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()][-200:]
    data = [p for p in data if p["Index"] in ixes]

    print(m)
    for s in ['NoReasons', 'Reasons']:
        print("\t", s, compute_acc_per_criteria(data, s))

print("\n========")
print("Finetuned")
print("========\n")

for m in ["dev-qwen-3-8b"]:
    data = [json.loads(l) for l in (open(f"finetuned_predictions/human/Geordie/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()][-200:]
    data = [p for p in data if p["Index"] in ixes]

    print(m)
    for s in ['NoReasons']:
        print("\t", s, compute_acc_per_criteria(data, s))

	 NoBreakdown (0.73, 0.68)
	 BreakdownNoReasons (0.76, 0.69)
	 BreakdownReasons (0.78, 0.71)
	 NoBreakdown (0.54, 0.56)
	 BreakdownNoReasons (0.65, 0.66)
	 BreakdownReasons (0.51, 0.55)
	 NoBreakdown (0.51, 0.54)
	 BreakdownNoReasons (0.54, 0.56)
	 BreakdownReasons (0.51, 0.53)
	 NoBreakdown (0.7, 0.62)
	 BreakdownNoReasons (0.7, 0.62)
	 BreakdownReasons (0.73, 0.64)
	 NoBreakdown (0.68, 0.64)
	 BreakdownNoReasons (0.73, 0.68)
	 BreakdownReasons (0.7, 0.62)

Finetuned

dev-qwen-3-8b
	 NoReasons (0.68, 0.64)
	 Reasons (0.7, 0.68)
dev-anthropic-claude-opus-4-1
	 NoReasons (0.51, 0.55)
	 Reasons (0.54, 0.57)
dev-gpt-5-mini
	 NoReasons (0.59, 0.62)
	 Reasons (0.57, 0.6)
dev-phi-4
	 NoReasons (0.76, 0.72)
	 Reasons (0.65, 0.65)
dev-gpt-41-longco-2025-04-14
	 NoReasons (0.65, 0.64)
	 Reasons (0.65, 0.65)

Finetuned

dev-qwen-3-8b
	 NoReasons (0.61, 0.47)


# Scoring

In [ ]:
from evaluation_utils import mcnemar_test, compute_gwet_ac1
import json
import numpy as np
from sklearn.metrics import f1_score

### Baseline (for all data)

In [ ]:
# Baseline (all data) perf

def boost(locale):
    #m = "dev-qwen-3-8b"
    #data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_all_crit.json", "r", encoding='utf-8')).readlines()]
    #random.shuffle(data)
    #ixes = [p["Index"] for p in data[-500:]]
    #train, test = data[:800], data[-200:]
    #train = [p for p in data if p["Index"] not in ixes]
    #test = [p for p in data if p["Index"] in ixes]

    preds = {}
    print(locale)
    for m in ["dev-qwen-3-8b", "dev-anthropic-claude-opus-4-1", "dev-gpt-5-mini", "dev-phi-4", "dev-gpt-41-longco-2025-04-14"]:
        data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_all_crit.json", "r", encoding='utf-8')).readlines()]
        #data = [p for p in data if p["Index"] in ixes]
        pred_data = [p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0 for p in data]
        preds[m] = pred_data
        if "gt" not in preds:
            preds["gt"] = [p["Label"] for p in data]
    print("Per-model test acc")
    avgs = {"F1": [], "Acc": []}
    for k, pred_array in preds.items():
        if k == "gt": continue
        gts = preds["gt"]
        after_acc = round(np.average([p == q for p, q in zip(gts, pred_array)]), 2)
        after_f1 = round(compute_gwet_ac1(pred_array, gts), 2) #round(f1_score(gts, pred_array, average="weighted"), 2)
        print(k, after_acc, after_f1)
        avgs["F1"].append(after_f1)
        avgs["Acc"].append(after_acc)

    print("===", np.average(avgs["F1"]), np.average(avgs["Acc"]))
    def rebalance(d):
        zeros = []
        ones = []
        for e in d:
            lab = e["NoBreakdown"]["response"]["Label"] if "Label" in e["NoBreakdown"]["response"] else 0
            if lab == 0: zeros.append(e)
            if lab == 1: ones.append(e)
        _d = zeros + ones[:len(zeros)]
        return _d


    preds = {}
    for m in ["dev-qwen-3-8b", "dev-anthropic-claude-opus-4-1", "dev-gpt-5-mini", "dev-phi-4", "dev-gpt-41-longco-2025-04-14"]:
        data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_all_crit.json", "r", encoding='utf-8')).readlines()]
        data = rebalance(data)
        #data = [p for p in data if p["Index"] in ixes]
        pred_data = [p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0 for p in data]
        preds[m] = pred_data
        if "gt" not in preds:
            preds[m + "_gt"] = [p["Label"] for p in data]

    print("Per-model test acc")
    avgs = {"F1": [], "Acc": []}
    for k, pred_array in preds.items():
        if "gt" in k: continue
        gts = preds[k + "_gt"]
        after_acc = round(np.average([p == q for p, q in zip(gts, pred_array)]), 2)
        after_f1 = round(compute_gwet_ac1(pred_array, gts), 2) #round(f1_score(gts, pred_array, average="weighted"), 2)
        print(k, after_acc, after_f1)
        avgs["F1"].append(after_f1)
        avgs["Acc"].append(after_acc)

    print("===", np.average(avgs["F1"]), np.average(avgs["Acc"]))
    ##print(mcnemar_test(preds_finetuned, preds_list, gts)[0])


In [ ]:
for loc in ["AAVE", "Yorkshire", "Cornish", "Geordie", "West_Frisian"]:
    boost(loc)
    print("")

AAVE


Per-model test acc
dev-qwen-3-8b 0.85 0.81
dev-anthropic-claude-opus-4-1 0.75 0.64
dev-gpt-5-mini 0.86 0.82
dev-phi-4 0.82 0.77
dev-gpt-41-longco-2025-04-14 0.91 0.89
=== 0.786 0.8379999999999999
Per-model test acc
dev-qwen-3-8b 0.61 0.29
dev-anthropic-claude-opus-4-1 0.57 0.25
dev-gpt-5-mini 0.64 0.34
dev-phi-4 0.55 0.22
dev-gpt-41-longco-2025-04-14 0.66 0.36
=== 0.292 0.6060000000000001

Yorkshire
Per-model test acc
dev-qwen-3-8b 0.87 0.84
dev-anthropic-claude-opus-4-1 0.71 0.6
dev-gpt-5-mini 0.87 0.85
dev-phi-4 0.62 0.43
dev-gpt-41-longco-2025-04-14 0.91 0.9
=== 0.724 0.796
Per-model test acc
dev-qwen-3-8b 0.54 0.22
dev-anthropic-claude-opus-4-1 0.53 0.22
dev-gpt-5-mini 0.55 0.22
dev-phi-4 0.52 0.2
dev-gpt-41-longco-2025-04-14 0.57 0.24
=== 0.22000000000000003 0.542

Cornish
Per-model test acc
dev-qwen-3-8b 0.54 0.12
dev-anthropic-claude-opus-4-1 0.6 0.21
dev-gpt-5-mini 0.54 0.29
dev-phi-4 0.54 0.13
dev-gpt-41-longco-2025-04-14 0.55 0.11
=== 0.172 0.554
Per-model test acc
dev-qwen-3

### Compare for the same test set as SFT/DPO

In [ ]:
def compare_and_score_strats(locale, filterit=False):

    m = "dev-qwen-3-8b"
    main_data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()]
    key = "Prompt" #"Prompt" if "Frisian" in locale else "Index"
    ixes = [p[key] + "#" + str(p["Output"]) for p in main_data]
    print(len(main_data))


    base_all_scores = {}
    data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_all_crit.json", "r", encoding='utf-8')).readlines()]
    data = [p for p in data if p[key]+  "#" + p["Output"] in ixes] # and p["Output"] in [q["Output"] for q in]]
    pred_data = [p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0 for p in data]
    gts = [p["Label"] for p in data]
    #print("Baseline Per-model test acc/f1/ac1")
    baseline_all_crit = pred_data[:]
    after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    after_f1 = round(f1_score(gts, pred_data)*100, 1)
    after_ac1 = round(compute_gwet_ac1(pred_data, gts)*100, 2)
    mcnemar = 0# round(mcnemar_test(baseline_all_crit[k], pred_array, gts)[-1], 2)
    base_all_scores["acc"] = after_acc
    base_all_scores["f1"] = after_f1
    base_all_scores["ac1"] = after_ac1
    print("Strategy 1", after_acc, " & ", after_f1," & ", after_ac1, mcnemar)
    data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_all_crit.json", "r", encoding='utf-8')).readlines()]
    data = [p for p in data if p[key]+ "#" + p["Output"] in ixes]
    pred_data = [p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0 for p in data]
    gts = [p["Label"] for p in data]
    #print("Finetuned all-crit test acc/f1/ac1")
    after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    after_f1 = round(f1_score(gts, pred_data)*100, 1)
    after_ac1 = round(compute_gwet_ac1(pred_data, gts)*100, 2)
    mcnemar = round(mcnemar_test(baseline_all_crit, pred_data, gts)[-1], 4)
    print("Strategy 1 (FT)", after_acc, " & ", after_f1," & ", after_ac1, mcnemar)
    print("Delta", " & ", round(- base_all_scores["acc"] + after_acc, 2), " & ", 
            round( - base_all_scores["f1"] + after_f1, 2), " & ", round( - base_all_scores["ac1"] + after_ac1, 2), " & ", mcnemar) #, " & ",after_ac1, mcnemar)

    ##################
    base_per_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5", "aggr"]}
    gts_per_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]}
    seen = {}
    _data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_per_crit.json", "r", encoding='utf-8')).readlines()]
    data = []
    for p in _data:
        if p[key]+  "#" + p["Output"] not in ixes: continue
        if p[key]+  "#" + p["Output"] not in seen: seen[p[key]+  "#" + p["Output"]] = -1
        if seen[p[key]+  "#" + p["Output"]] == 0: continue
        data.append(p)
        seen[p[key]+  "#" + p["Output"]] = 0

    #data = [p for p in data if p[key]+  "#" + p["Output"] in ixes] # and p["Output"] in [q["Output"] for q in]]
    #seen = {p[key]+  "#" + p["Output"]: 0 for p in data}
    pred_data = []
    for p in data:
        lab = []
        for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
            lab.append(p["NoReasons"][c]["response"])
            base_per_scores[c].append(lab[-1])
            gts_per_scores[c].append(p["Rubric"][c])
        pred_data.append( int( sum(lab) >= 6) )
        base_per_scores["aggr"].append(pred_data[-1])

    gts = [p["Label"] for p in data]
    #print("Baseline Per-crit test acc/f1/ac1")
    baseline_per_crit = pred_data[:]
    after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    after_f1 = round(f1_score(gts, pred_data)*100, 1)
    after_ac1 = round(compute_gwet_ac1(pred_data, gts)*100, 2)
    base_per_scores["acc"] = after_acc
    base_per_scores["f1"] = after_f1
    base_per_scores["ac1"] = after_ac1
    mcnemar = 0# round(mcnemar_test(baseline_all_crit[k], pred_array, gts)[-1], 2)
    for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
        base_per_scores[c + "_acc"] = round(np.average([p == q for p, q in zip(gts_per_scores[c], base_per_scores[c])])*100, 1)
        base_per_scores[c + "_f1"] = round(f1_score(gts_per_scores[c], base_per_scores[c])*100, 1)
        base_per_scores[c + "_ac1"] = round(compute_gwet_ac1(base_per_scores[c], gts_per_scores[c])*100, 2)
    print("Strategy 4 ", after_acc," & ", after_f1, " & ",after_ac1, mcnemar)


    remap = {"c1": "c1", "c2a": "c2", "c2b": "c3", "c3": "c4", "c4": "c5", "c5": "c6"}

    current_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5", "aggr"]}
    gts_per_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]}
    _data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()]
    data = []
    for p in _data:
        if p[key]+  "#" + p["Output"] not in seen: continue
        if seen[p[key]+  "#" + p["Output"]] == 1: continue
        data.append(p)
        seen[p[key]+  "#" + p["Output"]] = 1
    pred_data = []
    for p in data:
        lab = []
        for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
            lab.append(p["NoReasons"][c]["response"])
            current_scores[c].append(lab[-1])
            gts_per_scores[c].append(p["Rubric"][c])
        pred_data.append( int( sum(lab) >= 6) )
        current_scores["aggr"].append(pred_data[-1])

    gts = [p["Label"] for p in data]
    print("Finetuned Per-crit test acc/f1/ac1")
    after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    after_f1 = round(f1_score(gts, pred_data)*100, 1)
    after_ac1 = round(compute_gwet_ac1(pred_data, gts)*100, 2)
    print(len(baseline_per_crit), len(pred_data))
    mcnemar = round(mcnemar_test(baseline_per_crit, pred_data, gts)[-1], 4)
    for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
        c_acc = round(np.average([p == q for p, q in zip(gts_per_scores[c], current_scores[c])])*100, 1)
        c_f1 = round(f1_score(gts_per_scores[c], current_scores[c])*100, 1)
        c_mcnemar = round(mcnemar_test(base_per_scores[c], current_scores[c], gts_per_scores[c])[-1], 4)
        c_ac1 = round(compute_gwet_ac1(current_scores[c], gts_per_scores[c])*100, 2)
        #print(remap[c], "& ", c_mcnemar, " & ", round( - base_per_scores[c + "_acc"] + c_acc, 2), " & ", 
        #round(- base_per_scores[c + "_f1"] + c_f1, 2), " & ", round( - base_per_scores[c + "_ac1"] + c_ac1, 2), "\\\\") #, " & ",after_ac1, mcnemar)
    
    print("Strategy 4 (FT) ", after_acc, " & ",after_f1, " & ",after_ac1, mcnemar)
    print("Delta", " & ", round( - base_per_scores["acc"] + after_acc, 2), " & ", 
          round(- base_per_scores["f1"] + after_f1, 2), " & ", round( - base_per_scores["ac1"] + after_ac1, 2), " & ", mcnemar) #, " & ",after_ac1, mcnemar)

    #print(np.average(scores, 1))


In [ ]:
for loc in ["AAVE", "Yorkshire", "Cornish", "Geordie", "West_Frisian"]:
    print("\n", loc, "\n")
    compare_and_score_strats(loc)


 AAVE 

203
Strategy 1 87.2  &  92.9  &  84.57 0
Strategy 1 (FT) 91.6  &  95.6  &  90.9 0.0003
Delta  &  4.4  &  2.7  &  6.33  &  0.0003
Strategy 4  85.2  &  91.8  &  81.86 0
Finetuned Per-crit test acc/f1/ac1
203 203
Strategy 4 (FT)  91.1  &  95.4  &  90.31 0.0
Delta  &  5.9  &  3.6  &  8.45  &  0.0

 Yorkshire 

202
Strategy 1 84.2  &  91.2  &  80.72 0
Strategy 1 (FT) 96.0  &  98.0  &  95.88 0.0
Delta  &  11.8  &  6.8  &  15.16  &  0.0
Strategy 4  85.6  &  92.1  &  82.78 0
Finetuned Per-crit test acc/f1/ac1
202 202
Strategy 4 (FT)  95.5  &  97.7  &  95.3 0.0
Delta  &  9.9  &  5.6  &  12.52  &  0.0

 Cornish 

203
Strategy 1 55.2  &  63.7  &  15.09 0
Strategy 1 (FT) 70.0  &  67.0  &  40.37 0.0001
Delta  &  14.8  &  3.3  &  25.28  &  0.0001
Strategy 4  55.7  &  64.3  &  16.21 0
Finetuned Per-crit test acc/f1/ac1
203 203
Strategy 4 (FT)  63.1  &  67.8  &  27.69 0.0044
Delta  &  7.4  &  3.5  &  11.48  &  0.0044

 Geordie 

203
Strategy 1 64.0  &  75.4  &  40.78 0
Strategy 1 (FT) 62.6  &

### Compare DPO

In [ ]:
def boost_sft(locale):
    m = "dev-qwen-3-8b"
    data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_all_crit-dpo.json", "r", encoding='utf-8')).readlines()]
    ixes = [p["Index"] for p in data]

    print(locale)
    data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_all_crit.json", "r", encoding='utf-8')).readlines()]
    #data = [p for p in data if p["Index"] in ixes]
    pred_data = [p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0 for p in data]
    baseline_per_crit = pred_data[:]
    gts = [p["Label"] for p in data]
    print("SFT test acc")
    sft_after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    sft_after_f1 = round(f1_score(gts, pred_data)*100, 1)
    sft_after_ac1 = round(compute_gwet_ac1(pred_data, gts), 2)
    print(sft_after_acc, "  ", sft_after_f1, "  ", sft_after_ac1, "  ")

    data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_all_crit-dpo.json", "r", encoding='utf-8')).readlines()]
    pred_data = [p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0 for p in data]
    gts = [p["Label"] for p in data]
    print("DPO test acc")
    after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    after_f1 = round(f1_score(gts, pred_data)*100, 1)
    after_ac1 = round(compute_gwet_ac1(pred_data, gts), 2)
    print(len(baseline_per_crit), len(pred_data), len(gts))
    mcnemar = round(mcnemar_test(baseline_per_crit, pred_data, gts)[-1], 4)
    print(after_acc, "  ", after_f1, "  ", after_ac1, "  ", mcnemar)
    print(after_acc - sft_after_acc, "  ", after_f1 - sft_after_f1, "  ", after_ac1 - sft_after_ac1, "  ", mcnemar)



In [ ]:
for loc in ["Cornish"]:
    boost_sft(loc)
    print("")

Cornish
SFT test acc
70.0    67.0    0.4   
DPO test acc
203 203 203
52.7    63.4    0.13    0.0001
-17.299999999999997    -3.6000000000000014    -0.27    0.0001



### Synthetic data eval

In [ ]:


def boost_syn(locale):
    m = "dev-qwen-3-8b"
    data = [json.loads(l) for l in (open(f"finetuned_predictions/synthetic/{locale}/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()]
    key = "Prompt"
    ixes = [p["Prompt"]+  "#" + p["Output"] for p in data]

    seen = {}
    _data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_per_crit.json", "r", encoding='utf-8')).readlines()]
    data = []
    for p in _data:
        if p[key]+  "#" + p["Output"] not in ixes: continue
        if p[key]+  "#" + p["Output"] not in seen: seen[p[key]+  "#" + p["Output"]] = -1
        if seen[p[key]+  "#" + p["Output"]] == 0: continue
        data.append(p)
        seen[p[key]+  "#" + p["Output"]] = 0

    #data = [p for p in data if p[key]+  "#" + p["Output"] in ixes] # and p["Output"] in [q["Output"] for q in]]
    #seen = {p[key]+  "#" + p["Output"]: 0 for p in data}
    pred_data = []
    for p in data:
        lab = []
        for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
            lab.append(p["NoReasons"][c]["response"])
            #base_per_scores[c].append(lab[-1])
            #gts_per_scores[c].append(p["Rubric"][c])
        pred_data.append( int( sum(lab) >= 6) )
        #base_per_scores["aggr"].append(pred_data[-1])

    base_per_scores = {}
    gts = [p["Label"] for p in data]
    #print("Baseline Per-crit test acc/f1/ac1")
    baseline_per_crit = pred_data[:]
    after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    after_f1 = round(f1_score(gts, pred_data)*100, 1)
    after_ac1 = round(compute_gwet_ac1(pred_data, gts)*100, 2)
    base_per_scores["acc"] = after_acc
    base_per_scores["f1"] = after_f1
    base_per_scores["ac1"] = after_ac1
    mcnemar = 0# round(mcnemar_test(baseline_all_crit[k], pred_array, gts)[-1], 2)
    #for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
    #    base_per_scores[c + "_acc"] = round(np.average([p == q for p, q in zip(gts_per_scores[c], base_per_scores[c])])*100, 1)
    #    base_per_scores[c + "_f1"] = round(f1_score(gts_per_scores[c], base_per_scores[c])*100, 1)
    #    base_per_scores[c + "_ac1"] = round(compute_gwet_ac1(base_per_scores[c], gts_per_scores[c])*100, 2)
    print("Baseline", after_acc," & ", after_f1, " & ",after_ac1, mcnemar)


    print(locale)
    pred_per_scores = {}
    _data = [json.loads(l) for l in (open(f"finetuned_predictions/synthetic/{locale}/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()]
    pred_data = []
    data = []
    for p in _data:
        if p[key]+  "#" + p["Output"] not in seen: continue
        if seen[p[key]+  "#" + p["Output"]] == 1: continue
        data.append(p)
        seen[p[key]+  "#" + p["Output"]] = 1


    for p in data:
        lab = []
        for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
            lab.append(p["NoReasons"][c]["response"])
            #base_per_scores[c].append(lab[-1])
            #gts_per_scores[c].append(p["Rubric"][c])
        pred_data.append( int( sum(lab) >= 6) )
        #base_per_scores["aggr"].append(pred_data[-1])

    gts = [p["Label"] for p in data]
    #print("Baseline Per-crit test acc/f1/ac1")
    after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
    after_f1 = round(f1_score(gts, pred_data)*100, 1)
    after_ac1 = round(compute_gwet_ac1(pred_data, gts)*100, 2)
    pred_per_scores["acc"] = after_acc
    pred_per_scores["f1"] = after_f1
    pred_per_scores["ac1"] = after_ac1
    print(len(baseline_per_crit), len(pred_data), len(gts))
    mcnemar = round(mcnemar_test(baseline_per_crit, pred_data, gts)[-1], 2)
    #for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
    #    base_per_scores[c + "_acc"] = round(np.average([p == q for p, q in zip(gts_per_scores[c], base_per_scores[c])])*100, 1)
    #    base_per_scores[c + "_f1"] = round(f1_score(gts_per_scores[c], base_per_scores[c])*100, 1)
    #    base_per_scores[c + "_ac1"] = round(compute_gwet_ac1(base_per_scores[c], gts_per_scores[c])*100, 2)
    print("Syn FT", after_acc," & ", after_f1, " & ",after_ac1, mcnemar)

    for k, v in pred_per_scores.items():
        print("Delta", k, round(v - base_per_scores[k], 2))


In [ ]:
for locale in ['Cornish', "Yorkshire", "West_Frisian"]:
    print(locale)
    boost_syn(locale)
    print("")

Cornish
Baseline 55.7  &  64.3  &  16.21 0
Cornish
203 203 203
Syn FT 46.3  &  60.9  &  5.81 0.0
Delta acc -9.4
Delta f1 -3.4
Delta ac1 -10.4

Yorkshire
Baseline 85.6  &  92.1  &  82.78 0
Yorkshire
202 202 202
Syn FT 93.1  &  96.4  &  92.5 0.0
Delta acc 7.5
Delta f1 4.3
Delta ac1 9.72

West_Frisian
Baseline 73.6  &  74.5  &  47.27 0
West_Frisian
197 197 197
Syn FT 77.2  &  80.7  &  55.79 0.0
Delta acc 3.6
Delta f1 6.2
Delta ac1 8.52



### Compare SFT + human with ICL

In [ ]:
def task_boost(locale, filterit=False):

    m = "dev-qwen-3-8b"
    main_data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()]
    key = "Prompt" #"Prompt" if "Frisian" in locale else "Index"
    ixes = [p[key] + "#" + str(p["Output"]) for p in main_data]
    print(len(main_data))

    TASKS = ['GSM8K', 'OpenCode', 'OpenOrca', 'WildChat', 'shp']

    def record(name, task, acc, f1, ac1, mc):
        pass
        #with open("tmp_df.json", "a", encoding="utf-8") as f:
        #    f.write(json.dumps({"Locale": locale, "Task": task, "Model": name, "Accuracy": acc, "F1": f1, "Gwet AC1": ac1})+ "\n")

    base_all_scores = {t: {} for t in TASKS}
    base_preds = {}
    task_preds = {t: [] for t in TASKS} | {t + "_gt": [] for t in TASKS}
    data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_all_crit.json", "r", encoding='utf-8')).readlines()]
    data = [p for p in data if p[key]+  "#" + p["Output"] in ixes] # and p["Output"] in [q["Output"] for q in]]
    for p in data:
        label = p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0
        gt = p["Label"]
        source = p["Source"]
        task_preds[source].append(label)
        task_preds[source + "_gt"].append(gt)

    print("Baseline Per-model test acc/f1/ac1")
    for t in TASKS:
        pred_data = task_preds[t] 
        base_preds[t] = pred_data
        gts = task_preds[t + "_gt"]
        after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
        after_f1 = round(f1_score(gts, pred_data)*100, 1)
        after_ac1 = round(compute_gwet_ac1(pred_data, gts), 2)
        mcnemar = 0 #round(mcnemar_test(baseline_all_crit[k], pred_array, gts)[-1], 2)
        base_all_scores[t]["acc"] = after_acc
        base_all_scores[t]["f1"] = after_f1
        base_all_scores[t]["ac1"] = after_ac1
        record("Strategy 1", t, after_acc, after_f1, after_ac1, mcnemar)
        print("  ", t, " & ", after_acc, " & ", after_f1," & ", after_ac1, " & ", mcnemar)

    # All crit
    task_preds = {t: [] for t in TASKS} | {t + "_gt": [] for t in TASKS}
    after_all_scores = {t: {} for t in TASKS}
    data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_all_crit.json", "r", encoding='utf-8')).readlines()]
    data = [p for p in data if p[key]+ "#" + p["Output"] in ixes]
    for p in data:
        label = p["NoBreakdown"]["response"]["Label"] if "Label" in p["NoBreakdown"]["response"] else 0
        gt = p["Label"]
        source = p["Source"]
        task_preds[source].append(label)
        task_preds[source + "_gt"].append(gt)

    print("Finetuned all-crit test acc/f1/ac1")
    for t in TASKS:
        pred_data = task_preds[t] 
        gts = task_preds[t + "_gt"]
        after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
        after_f1 = round(f1_score(gts, pred_data)*100, 1)
        after_ac1 = round(compute_gwet_ac1(pred_data, gts), 2)
        after_all_scores[t]["acc"] = after_acc
        after_all_scores[t]["f1"] = after_f1
        after_all_scores[t]["ac1"] = after_ac1
        mcnemar = round(mcnemar_test(base_preds[t], pred_data, gts)[-1], 4)
        record("Strategy 1 (FT)", t, after_acc, after_f1, after_ac1, mcnemar)
 
        print(t, " & ", after_acc, " & ", after_f1," & ", after_ac1, " & ", mcnemar)

    print("Deltas")
    deltas = {}
    for t in TASKS:
        before = base_all_scores[t] 
        after = after_all_scores[t]
        after_acc = round(after["acc"] - before["acc"], 1)
        after_f1 = round(after["f1"] - before["f1"], 1)
        after_ac1 = round(after["ac1"] - before["ac1"], 2)
        mcnemar = 0 #round(mcnemar_test(baseline_all_crit, pred_data, gts)[-1], 4)
        deltas[t] = {
            "acc": after_acc,
            "f1": after_f1, 
            "ac1": after_ac1,
            "acc_g": after["acc"],
            "f1_g": after["f1"], 
            "ac1_g": after["ac1"]
        }
        print(t, " & ", after_acc, " & ", after_f1," & ", after_ac1, " & ", mcnemar)
    #return deltas

    ##################
    base_per_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5", "aggr"]}
    gts_per_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]}

    task_preds = {t: [] for t in TASKS} | {t + "_gt": [] for t in TASKS}
    after_all_scores = {t: {} for t in TASKS}
    base_all_scores = {}
    bases_per_scores = {t: {} for t in TASKS}

    seen = {}
    _data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_per_crit.json", "r", encoding='utf-8')).readlines()]
    data = []
    for p in _data:
        if p[key]+  "#" + p["Output"] not in ixes: continue
        if p[key]+  "#" + p["Output"] not in seen: seen[p[key]+  "#" + p["Output"]] = -1
        if seen[p[key]+  "#" + p["Output"]] == 0: continue
        data.append(p)
        seen[p[key]+  "#" + p["Output"]] = 0

    pred_data = []
    for p in data:
        lab = []
        for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
            lab.append(p["NoReasons"][c]["response"])
            base_per_scores[c].append(lab[-1])
            gts_per_scores[c].append(p["Rubric"][c])
        label = int( sum(lab) >= 6)
        pred_data.append( label )
        base_per_scores["aggr"].append(pred_data[-1])
        task_preds[p["Source"]].append(label)
        task_preds[p["Source"] + "_gt"].append(p["Label"])

    #gts = [p["Label"] for p in data]
    print("Baseline Per-crit test acc/f1/ac1")
    #baseline_per_crit = pred_data[:]
    for t in TASKS:
        gts = task_preds[t + "_gt"]
        pred_data = task_preds[t]
        base_all_scores[t] = pred_data
        after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
        after_f1 = round(f1_score(gts, pred_data)*100, 1)
        after_ac1 = round(compute_gwet_ac1(pred_data, gts), 2)
        bases_per_scores[t]["acc"] = after_acc
        bases_per_scores[t]["f1"] = after_f1
        bases_per_scores[t]["ac1"] = after_ac1
        mcnemar = 0
        record("Strategy 4", t, after_acc, after_f1, after_ac1, mcnemar)
        print(t, " & ", after_acc, " & ", after_f1," & ", after_ac1, " & ", mcnemar)

    #mcnemar = 0# round(mcnemar_test(baseline_all_crit[k], pred_array, gts)[-1], 2)
    #for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
    #    base_per_scores[c + "_acc"] = round(np.average([p == q for p, q in zip(gts_per_scores[c], base_per_scores[c])])*100, 1)
    #    base_per_scores[c + "_f1"] = round(f1_score(gts_per_scores[c], base_per_scores[c])*100, 1)
    #    base_per_scores[c + "_ac1"] = round(compute_gwet_ac1(base_per_scores[c], gts_per_scores[c])*100, 2)
    #print(after_acc," & ", after_f1, " & ",after_ac1, mcnemar)

    remap = {"c1": "c1", "c2a": "c2", "c2b": "c3", "c3": "c4", "c4": "c5", "c5": "c6"}

    current_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5", "aggr"]}
    gts_per_scores = {c: [] for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]}
    task_preds = {t: [] for t in TASKS} | {t + "_gt": [] for t in TASKS}
    after_all_scores = {t: {} for t in TASKS}

    _data = [json.loads(l) for l in (open(f"finetuned_predictions/human/{locale}/{m}_0_per_crit.json", "r", encoding='utf-8')).readlines()]
    data = []
    for p in _data:
        if p[key]+  "#" + p["Output"] not in seen: continue
        if seen[p[key]+  "#" + p["Output"]] == 1: continue
        data.append(p)
        seen[p[key]+  "#" + p["Output"]] = 1
    pred_data = []
    for p in data:
        lab = []
        for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
            lab.append(p["NoReasons"][c]["response"])
            current_scores[c].append(lab[-1])
            gts_per_scores[c].append(p["Rubric"][c])
        label = int( sum(lab) >= 6)
        pred_data.append( label )
        current_scores["aggr"].append(pred_data[-1])
        task_preds[p["Source"]].append(label)
        task_preds[p["Source"] + "_gt"].append(p["Label"])

    #gts = [p["Label"] for p in data]
    print("Finetuned Per-crit test acc/f1/ac1")
    for t in TASKS:
        gts = task_preds[t + "_gt"]
        pred_data = task_preds[t]
        after_acc = round(np.average([p == q for p, q in zip(gts, pred_data)])*100, 1)
        after_f1 = round(f1_score(gts, pred_data)*100, 1)
        after_ac1 = round(compute_gwet_ac1(pred_data, gts), 2)
        after_all_scores[t]["acc"] = after_acc
        after_all_scores[t]["f1"] = after_f1
        after_all_scores[t]["ac1"] = after_ac1

        #print(len(baseline_per_crit), len(pred_data))
        mcnemar = round(mcnemar_test(base_all_scores[t], pred_data, gts)[-1], 4)
        record("Strategy 4 (FT)", t, after_acc, after_f1, after_ac1, mcnemar)
        print(t, " & ", after_acc, " & ", after_f1," & ", after_ac1, " & ", mcnemar)

    print("Deltas")
    _deltas = {}
    for t in TASKS:
        before = bases_per_scores[t] 
        after = after_all_scores[t]
        after_acc = round(after["acc"] - before["acc"], 1)
        after_f1 = round(after["f1"] - before["f1"], 1)
        after_ac1 = round(after["ac1"] - before["ac1"], 2)
        mcnemar = 0 #round(mcnemar_test(baseline_all_crit, pred_data, gts)[-1], 4)
        _deltas[t] = {
            "acc": after_acc,
            "f1": after_f1, 
            "ac1": after_ac1,
            "acc_g": after["acc"],
            "f1_g": after["f1"], 
            "ac1_g": after["ac1"]
        }
        print(t, " & ", after_acc, " & ", after_f1," & ", after_ac1, " & ", mcnemar)


    #for c in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
    #    c_acc = round(np.average([p == q for p, q in zip(gts_per_scores[c], current_scores[c])])*100, 1)
    #    c_f1 = round(f1_score(gts_per_scores[c], current_scores[c])*100, 1)
    #    c_mcnemar = round(mcnemar_test(base_per_scores[c], current_scores[c], gts_per_scores[c])[-1], 4)
    #    c_ac1 = round(compute_gwet_ac1(current_scores[c], gts_per_scores[c])*100, 2)
    #    #print(remap[c], "& ", c_mcnemar, " & ", round( - base_per_scores[c + "_acc"] + c_acc, 2), " & ", 
    #    #round(- base_per_scores[c + "_f1"] + c_f1, 2), " & ", round( - base_per_scores[c + "_ac1"] + c_ac1, 2), "\\\\") #, " & ",after_ac1, mcnemar)
    #print("Aggregate", " & ", mcnemar, " & ", round( - base_per_scores["acc"] + after_acc, 2), " & ", 
    #      round(- base_per_scores["f1"] + after_f1, 2), " & ", round( - base_per_scores["ac1"] + after_ac1, 2)) #, " & ",after_ac1, mcnemar)

    #print(after_acc, " & ",after_f1, " & ",after_ac1, mcnemar)
    #print(np.average(scores, 1))

    return deltas


In [ ]:
deltas_avg = {}

for loc in ["AAVE", "Cornish", "Geordie", "Yorkshire", "West_Frisian"]:
    print("\n", loc, "\n")
    delta = task_boost(loc)
    for k, d in delta.items():
        if k not in deltas_avg: deltas_avg[k] = {"acc": [], "f1": [], "ac1": [], "acc_g": [], "f1_g": [], "ac1_g": [],}
        for m, v in d.items():
            deltas_avg[k][m].append(v)

print("----------")
for k, d in deltas_avg.items():
    print(" ")
    for m, v in d.items():
        if "_g" in m: continue
        print(k, "   ", m, "  ", round(np.average(v), 2), " ; ", round(np.average(d[m + "_g"]), 2))


 AAVE 



203
Baseline Per-model test acc/f1/ac1
   GSM8K  &  90.0  &  94.7  &  0.89  &  0
   OpenCode  &  70.8  &  80.6  &  0.53  &  0
   OpenOrca  &  98.0  &  98.9  &  0.98  &  0
   WildChat  &  92.6  &  96.2  &  0.92  &  0
   shp  &  86.2  &  92.6  &  0.84  &  0
Finetuned all-crit test acc/f1/ac1
GSM8K  &  94.0  &  96.9  &  0.94  &  0.25
OpenCode  &  87.5  &  93.3  &  0.86  &  0.0007
OpenOrca  &  95.9  &  97.9  &  0.96  &  0.5
WildChat  &  92.6  &  96.2  &  0.92  &  1.0
shp  &  86.2  &  92.6  &  0.84  &  1.0
Deltas
GSM8K  &  4.0  &  2.2  &  0.05  &  0
OpenCode  &  16.7  &  12.7  &  0.33  &  0
OpenOrca  &  -2.1  &  -1.0  &  -0.02  &  0
WildChat  &  0.0  &  0.0  &  0.0  &  0
shp  &  0.0  &  0.0  &  0.0  &  0


Baseline Per-crit test acc/f1/ac1
GSM8K  &  90.0  &  94.7  &  0.89  &  0
OpenCode  &  60.4  &  72.5  &  0.34  &  0
OpenOrca  &  98.0  &  98.9  &  0.98  &  0
WildChat  &  96.3  &  98.0  &  0.96  &  0
shp  &  86.2  &  92.6  &  0.84  &  0
Finetuned Per-crit test acc/f1/ac1
GSM8K  &  92.0  &  95.8  &  0.91  &  0.625
OpenCode  &  87.5  &  93.3  &  0.86  &  0.0009
OpenOrca  &  95.9  &  97.9  &  0.96  &  0.5
WildChat  &  92.6  &  96.2  &  0.92  &  0.5
shp  &  86.2  &  92.6  &  0.84  &  1.0
Deltas
GSM8K  &  2.0  &  1.1  &  0.02  &  0
OpenCode  &  27.1  &  20.8  &  0.52  &  0
OpenOrca  &  -2.1  &  -1.0  &  -0.02  &  0
WildChat  &  -3.7  &  -1.8  &  -0.04  &  0
shp  &  0.0  &  0.0  &  0.0  &  0

 Cornish 

203


Baseline Per-model test acc/f1/ac1
   GSM8K  &  53.8  &  66.7  &  0.2  &  0
   OpenCode  &  45.7  &  24.2  &  -0.01  &  0
   OpenOrca  &  52.9  &  64.7  &  0.15  &  0
   WildChat  &  70.8  &  77.4  &  0.46  &  0
   shp  &  63.3  &  76.6  &  0.44  &  0
Finetuned all-crit test acc/f1/ac1
GSM8K  &  69.2  &  69.2  &  0.38  &  0.0186
OpenCode  &  80.4  &  30.8  &  0.74  &  0.0041
OpenOrca  &  62.7  &  57.8  &  0.27  &  0.0639
WildChat  &  79.2  &  82.8  &  0.6  &  0.0074
shp  &  60.0  &  73.9  &  0.38  &  0.625
Deltas
GSM8K  &  15.4  &  2.5  &  0.18  &  0
OpenCode  &  34.7  &  6.6  &  0.75  &  0
OpenOrca  &  9.8  &  -6.9  &  0.12  &  0
WildChat  &  8.4  &  5.4  &  0.14  &  0
shp  &  -3.3  &  -2.7  &  -0.06  &  0
Baseline Per-crit test acc/f1/ac1
GSM8K  &  51.9  &  65.8  &  0.17  &  0
OpenCode  &  54.3  &  36.4  &  0.15  &  0
OpenOrca  &  51.0  &  63.8  &  0.13  &  0
WildChat  &  70.8  &  77.4  &  0.46  &  0
shp  &  60.0  &  73.9  &  0.38  &  0
Finetuned Per-crit test acc/f1/ac1
GSM8K  &  59

Baseline Per-crit test acc/f1/ac1
GSM8K  &  68.9  &  79.4  &  0.51  &  0
OpenCode  &  54.9  &  63.5  &  0.15  &  0
OpenOrca  &  54.5  &  70.6  &  0.3  &  0
WildChat  &  62.9  &  76.4  &  0.44  &  0
shp  &  71.4  &  83.3  &  0.62  &  0
Finetuned Per-crit test acc/f1/ac1
GSM8K  &  62.2  &  76.7  &  0.46  &  0.6875
OpenCode  &  54.9  &  70.1  &  0.28  &  1.0
OpenOrca  &  54.5  &  70.6  &  0.3  &  1.0
WildChat  &  65.7  &  78.6  &  0.5  &  0.625
shp  &  71.4  &  83.3  &  0.62  &  1.0
Deltas
GSM8K  &  -6.7  &  -2.7  &  -0.05  &  0
OpenCode  &  0.0  &  6.6  &  0.13  &  0
OpenOrca  &  0.0  &  0.0  &  0.0  &  0
WildChat  &  2.8  &  2.2  &  0.06  &  0
shp  &  0.0  &  0.0  &  0.0  &  0

 Yorkshire 

202


Baseline Per-model test acc/f1/ac1
   GSM8K  &  87.2  &  93.2  &  0.86  &  0
   OpenCode  &  60.4  &  72.7  &  0.34  &  0
   OpenOrca  &  90.4  &  94.9  &  0.89  &  0
   WildChat  &  100.0  &  100.0  &  1.0  &  0
   shp  &  100.0  &  100.0  &  1.0  &  0
Finetuned all-crit test acc/f1/ac1
GSM8K  &  97.9  &  98.9  &  0.98  &  0.0703
OpenCode  &  92.5  &  96.1  &  0.92  &  0.0
OpenOrca  &  94.2  &  97.0  &  0.94  &  0.25
WildChat  &  100.0  &  100.0  &  1.0  &  1.0
shp  &  100.0  &  100.0  &  1.0  &  1.0
Deltas
GSM8K  &  10.7  &  5.7  &  0.12  &  0
OpenCode  &  32.1  &  23.4  &  0.58  &  0
OpenOrca  &  3.8  &  2.1  &  0.05  &  0
WildChat  &  0.0  &  0.0  &  0.0  &  0
shp  &  0.0  &  0.0  &  0.0  &  0


Baseline Per-crit test acc/f1/ac1
GSM8K  &  87.2  &  93.2  &  0.86  &  0
OpenCode  &  66.0  &  77.5  &  0.46  &  0
OpenOrca  &  90.4  &  94.9  &  0.89  &  0
WildChat  &  100.0  &  100.0  &  1.0  &  0
shp  &  100.0  &  100.0  &  1.0  &  0
Finetuned Per-crit test acc/f1/ac1
GSM8K  &  97.9  &  98.9  &  0.98  &  0.0703
OpenCode  &  90.6  &  94.9  &  0.89  &  0.0003
OpenOrca  &  94.2  &  97.0  &  0.94  &  0.25
WildChat  &  100.0  &  100.0  &  1.0  &  1.0
shp  &  100.0  &  100.0  &  1.0  &  1.0
Deltas
GSM8K  &  10.7  &  5.7  &  0.12  &  0
OpenCode  &  24.6  &  17.4  &  0.43  &  0
OpenOrca  &  3.8  &  2.1  &  0.05  &  0
WildChat  &  0.0  &  0.0  &  0.0  &  0
shp  &  0.0  &  0.0  &  0.0  &  0

 West_Frisian 

203


Baseline Per-model test acc/f1/ac1
   GSM8K  &  84.4  &  89.8  &  0.76  &  0
   OpenCode  &  70.2  &  65.3  &  0.42  &  0
   OpenOrca  &  76.7  &  85.7  &  0.67  &  0
   WildChat  &  57.7  &  35.3  &  0.24  &  0
   shp  &  84.2  &  87.0  &  0.7  &  0
Finetuned all-crit test acc/f1/ac1
GSM8K  &  90.6  &  93.6  &  0.85  &  0.0039
OpenCode  &  57.9  &  33.3  &  0.26  &  0.2863
OpenOrca  &  79.1  &  85.7  &  0.66  &  0.2101
WildChat  &  92.3  &  75.0  &  0.9  &  0.0
shp  &  63.2  &  74.1  &  0.37  &  1.0
Deltas
GSM8K  &  6.2  &  3.8  &  0.09  &  0
OpenCode  &  -12.3  &  -32.0  &  -0.16  &  0
OpenOrca  &  2.4  &  0.0  &  -0.01  &  0
WildChat  &  34.6  &  39.7  &  0.66  &  0
shp  &  -21.0  &  -12.9  &  -0.33  &  0
Baseline Per-crit test acc/f1/ac1
GSM8K  &  83.3  &  88.9  &  0.73  &  0
OpenCode  &  66.1  &  55.8  &  0.36  &  0
OpenOrca  &  78.6  &  86.6  &  0.68  &  0
WildChat  &  68.0  &  42.9  &  0.46  &  0
shp  &  84.2  &  85.7  &  0.69  &  0
Finetuned Per-crit test acc/f1/ac1
GSM8K  &  8

# Ensembles

In [ ]:
def get_human_data(ixes, locale):

    train_preds, test_preds = {}, {}
    ms = ["dev-qwen-3-8b", "dev-anthropic-claude-opus-4-1", "dev-gpt-5-mini",
          "dev-phi-4", "dev-gpt-41-longco-2025-04-14"]

    for m in [ms[0]]:
        data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_per_crit.json", "r", encoding='utf-8')).readlines()]
        train_data = [p for p in data if p["Index"] in ixes[:500]]
        test_data = [p for p in data if p["Index"] in ixes[500:]]
        pred_data = [
                    [p["Rubric"][c] for c in ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']]
            for p in train_data]
        train_preds["human"] = pred_data
        pred_data = [
                    [p["Rubric"][c] for c in ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']]
            for p in test_data]
        test_preds["human"] = pred_data

        if "gt" not in train_preds: 
            train_preds["gt"] = [p["Label"] for p in train_data]
        if "gt" not in test_preds: 
            test_preds["gt"] = [p["Label"] for p in test_data]

    return train_preds, test_preds


def get_criterion_data(crit, ixes, locale, print_acc=False):

    train_preds, test_preds = {}, {}
    ms = ["dev-qwen-3-8b", 
          "dev-anthropic-claude-opus-4-1", 
          "dev-gpt-5-mini",
          "dev-phi-4", 
          "dev-gpt-41-longco-2025-04-14"]

    for m in ms:

        data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_per_crit.json", "r", encoding='utf-8')).readlines()]
        #random.shuffle(data)
        train_data = [p for p in data if p["Index"] in ixes[:500]]
        test_data = [p for p in data if p["Index"] in ixes[500:]]
        #data = [p for p in data if p["Index"] in ixes]
        pred_data = [
                    [p["Reasons"][c]["response"] if p["Reasons"][c]["response"] in [0, 1] else 0 for c in [crit]] #, 'c2a', ]]#'c2b', 'c3', 'c4', 'c5']]
                    #[p["Rubric"][c] for c in ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']]
            for p in train_data]
        train_preds[m] = pred_data
        pred_data = [
                    [p["Reasons"][c]["response"] if p["Reasons"][c]["response"] in [0, 1] else 0 for c in [crit]  ]#, 'c2a',]]# 'c2b', 'c3', 'c4', 'c5']]
                    #[p["Rubric"][c] for c in ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']]
            for p in test_data]
        test_preds[m] = pred_data

        if "gt" not in train_preds: 
            train_preds["gt"] = [p["Rubric"][crit] for p in train_data] #[p["Label"] for p in train_data]
        if "gt" not in test_preds: 
            test_preds["gt"] = [p["Rubric"][crit] for p in test_data]


    if print_acc:
        for k, v in test_preds.items():
            if k == "gt": continue
            _v = [w[0] for w in v]
            acc = np.average([p == q for p, q in zip(_v, test_preds["gt"])])
            f1 = f1_score(test_preds["gt"], v)
            print("\t", k, crit, round(float(acc), 2), round(f1, 2))

    return train_preds, test_preds

def get_transpose(dataset):
    transpose = []
    for i in range(len(dataset["gt"])):
        row = []
        for k, v in dataset.items():
            if k != "gt": row += v[i] 
        transpose.append(row)
    return transpose


def get_label(arr, threshold=6):
    if sum(arr) >= threshold: return 1
    return 0


def boost_for_crit(locale, crit, ixes):

    train_preds, test_preds = get_criterion_data(crit, ixes, locale)
    
    clf = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2), n_estimators=100, 
                             learning_rate=0.1, random_state=123)

    train_m = get_transpose(train_preds)
    train_labels = [1 if get_label(d, 3) == p else 0 for d, p in zip(train_m, train_preds["gt"])]
    clf.fit(train_m, train_labels)

    test_m = get_transpose(test_preds)
    test_labels = [1 if get_label(d, 3) == p else 0 for d, p in zip(test_m, test_preds["gt"])]

    b_preds = clf.predict(test_m)

    acc = np.average([p == q for p, q in zip(b_preds, test_labels)])
    f1 = f1_score(test_labels, b_preds)

    print("Test avg for", crit, round(float(acc), 2), round(f1, 2))
    return b_preds, clf


def score_crit_vectors(crit_vectors, ground_truth):
    labels = []
    for i in range(len(ground_truth)):
        tmp_array = [v[i] for v in crit_vectors]
        labels.append(get_label(tmp_array))
    
    acc = np.average([p == q for p, q in zip(labels, ground_truth)])
    f1 = f1_score(ground_truth, labels)

    print("Final test_avg", round(float(acc), 2), round(f1, 2))



from sklearn.ensemble import AdaBoostClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier


def do_ensemble(classifiers, ixes, locale):

    train_preds, test_preds = get_human_data(ixes, locale)
    clf = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2), n_estimators=100, 
                             learning_rate=0.1, random_state=123)

    train_m = []
    for i in range(len(train_preds["gt"])):
        row = [
            classifiers[j](v[j]) for j, v in enumerate(train_preds["human"][i]) 
        ]
        #row = [v[i] for k, v in train_preds.items() if k != "gt"]
        train_m.append(row)
    clf.fit(train_m, train_preds["gt"])

    test_m = []
    for i in range(len(test_preds["gt"])):
        row = []
        for k, v in test_preds.items():
            if k != "gt":
                row += v[i]
        #row = [v[i] for k, v in train_preds.items() if k != "gt"]
        test_m.append(row)

    b_preds = clf.predict(test_m)
    acc = np.average([p == q for p, q in zip(b_preds, test_preds["gt"])])
    f1 = f1_score(test_preds["gt"], b_preds)

    print("Test avg for", round(float(acc), 2), round(f1, 2))
    return b_preds, clf

In [ ]:
for c in ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']:
    print("c", c)
    _ = get_criterion_data(c, ixes, "Geordie", print_acc=True)

c c1
	 dev-qwen-3-8b c1 0.89 0.94
	 dev-gpt-5-mini c1 0.95 0.98
	 dev-phi-4 c1 0.92 0.96
	 dev-gpt-41-longco-2025-04-14 c1 0.92 0.96
c c2a
	 dev-qwen-3-8b c2a 0.73 0.84
	 dev-gpt-5-mini c2a 0.72 0.83
	 dev-phi-4 c2a 0.71 0.82
	 dev-gpt-41-longco-2025-04-14 c2a 0.75 0.85
c c2b
	 dev-qwen-3-8b c2b 0.88 0.94
	 dev-gpt-5-mini c2b 0.88 0.93
	 dev-phi-4 c2b 0.9 0.95
	 dev-gpt-41-longco-2025-04-14 c2b 0.91 0.95
c c3
	 dev-qwen-3-8b c3 0.7 0.82
	 dev-gpt-5-mini c3 0.7 0.81
	 dev-phi-4 c3 0.65 0.78
	 dev-gpt-41-longco-2025-04-14 c3 0.72 0.84
c c4
	 dev-qwen-3-8b c4 0.95 0.98
	 dev-gpt-5-mini c4 0.98 0.99
	 dev-phi-4 c4 0.9 0.95
	 dev-gpt-41-longco-2025-04-14 c4 0.96 0.98
c c5
	 dev-qwen-3-8b c5 0.87 0.93
	 dev-gpt-5-mini c5 0.88 0.94
	 dev-phi-4 c5 0.76 0.86
	 dev-gpt-41-longco-2025-04-14 c5 0.91 0.95


In [ ]:
vecs = []
ensemble = []
# Each clf is trained with the LLM score and the correct label.
# We could interpret this as 'we trained a judge which predicts labels from LLM scores',
# or we could interpret it as 'our judge can tell what is the possible label given this LLM noise' 
# Hence, instead of checking whether LLM score == label, we interpret it as LLM score => correct
for c in ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']:
    v, clf = boost_for_crit("Geordie", c, ixes)
    vecs.append(v)
    ensemble.append(clf)

train_dict, test_dict = get_human_data(ixes, "Geordie")
# This reproduces the expected score from the LLMs
score_crit_vectors(vecs, test_dict["gt"])

Test avg for c1 0.98 0.99
Test avg for c2a 0.74 0.85
Test avg for c2b 0.96 0.98
Test avg for c3 0.72 0.83
Test avg for c4 0.98 0.99
Test avg for c5 0.92 0.96
Final test_avg 0.61 0.73


In [ ]:
# Now we'll predict the real score by predicting their correctness
locale = "Geordie"

fixed_vectors = []


train_pred_data, test_pred_data = get_criterion_data("c1", ixes, locale)
final_labels = [[] for _ in range(len(test_pred_data["gt"]))]

wrongs = {}
for k, (clf, crit, vec_crit) in enumerate(zip(ensemble, ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5'], vecs)):
    # passed in [0, 0, ...] and got [0, 1]
    # CAREFUL: test_preds[gt] is the HUMAN label.
    train_pred_data, test_pred_data = get_criterion_data(crit, ixes, locale)
    test_features = get_transpose(test_pred_data) # Does not return gt

    new_vec_crit = []
    for prediction, features in zip(vec_crit, test_features):
        # Label crit by majority vote
        if crit == "c3": new_vec_crit.append(1); continue
        if prediction == 1:
            new_vec_crit.append(get_label(features, 4))
        else: # It is wrong; we flip
            label = get_label(features, 4)
            if label == 0: 
                new_vec_crit.append(1)
            if label == 1: 
                new_vec_crit.append(0)

    for i in range(len(new_vec_crit)):
        final_labels[i].append(new_vec_crit[i])

    acc = np.average([p == q for p, q in zip(new_vec_crit, test_pred_data["gt"])])
    f1 = f1_score(test_pred_data["gt"], new_vec_crit)
    wrongs[crit] = [(i, vec_crit) for i in range(len(new_vec_crit)) if new_vec_crit[i] != test_pred_data["gt"][i]]
    print(crit, round(float(acc), 2), round(f1, 2))
    
_, test_preds = get_human_data(ixes, locale)
b_preds = [get_label(x) for x in final_labels]

acc = np.average([p == q for p, q in zip(b_preds, test_preds["gt"])])
wrongs["final"] = [i for i in range(len(b_preds)) if b_preds[i] != test_preds["gt"][i]]

f1 = f1_score(test_preds["gt"], b_preds)

print("Test avg", round(float(acc), 2), round(f1, 2))

c1 0.92 0.96
c2a 0.7 0.81
c2b 0.88 0.94
c3 0.73 0.84
c4 0.93 0.96
c5 0.86 0.92
Test avg 0.6 0.69


## Voting

In [ ]:
def boost(locale):
    m = "dev-qwen-3-8b"
    data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_all_crit.json", "r", encoding='utf-8')).readlines()]
    #random.shuffle(data)
    #ixes = [p["Index"] for p in data[-500:]]
    #train, test = data[:800], data[-200:]
    #train = [p for p in data if p["Index"] not in ixes]
    #test = [p for p in data if p["Index"] in ixes]

    #preds = {}
    #for m in ["dev-qwen-3-8b", 
    ##"dev-anthropic-claude-opus-4-1",
    #         "dev-gpt-5-mini",
    #        "dev-phi-4", "dev-gpt-41-longco-2025-04-14"]:
    #    data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_all_crit.json", "r", encoding='utf-8')).readlines()]
    #    #data = [p for p in data if p["Index"] in ixes]
    #    pred_data = [p["BreakdownNoReasons"]["response"]["Label"] if "Label" in p["BreakdownNoReasons"]["response"] else 0 for p in data]
    #    preds[m] = pred_data
    #    if "gt" not in preds:
    #        preds["gt"] = [p["Label"] for p in data]
    #print("Per-model test acc")
    #for k, pred_array in preds.items():
    #    if k == "gt": continue
    #    gts = preds["gt"]
    #    after_acc = round(np.average([p == q for p, q in zip(gts, pred_array)]), 2)
    #    after_f1 = round(f1_score(gts, pred_array, average="weighted"), 2)
    #    print(k, after_acc, after_f1)

    _preds = {}
    preds = {}
    for m in ["dev-qwen-3-8b", "dev-anthropic-claude-opus-4-1", "dev-gpt-5-mini",
            "dev-phi-4", "dev-gpt-41-longco-2025-04-14"]:

        data = [json.loads(l) for l in (open(f"icl_predictions/{locale}/{m}_5_per_crit.json", "r", encoding='utf-8')).readlines()]
        #data = [p for p in data if p["Index"] in ixes]
        ixes = [p["Index"]]
        pred_data = [
                    [p["NoReasons"][c]["response"] for c in ['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']]
             for p in data]
        preds[m] = pred_data
        preds[m + "_gt"] = [p["Label"] for p in data]


    #print("Per-model perf")
    avgs = {}
    for t in [0, 1, 2, 3, 4]:
        for k, _pred_array in preds.items():
            if "gt" in k: continue
            gts = preds[k + "_gt"]
            pred_array = [int(sum(q) >= len(['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']) - t) for q in _pred_array]
            after_acc = round(np.average([p == q for p, q in zip(gts, pred_array)])*100, 1)
            after_f1 = round(f1_score(gts, pred_array, average="weighted")*100, 1)
            #print(t, k, after_acc, after_f1)

            if t not in avgs: avgs[t] = {"acc": [], "f1": []}
            avgs[t]["acc"].append(after_acc)
            avgs[t]["f1"].append(after_f1)
        #print(t, "acc", round(np.average(avgs[t]["acc"]), 1), "f1", round(np.average(avgs[t]["f1"]), 1))

    print(0, "acc", round(np.average(avgs[0]["acc"]), 1), "f1", round(np.average(avgs[0]["f1"]), 1))
    ##print("Average per model", after_acc, after_f1)
    for t in [0, 1, 2]:
        preds_list = []
        gts = []
        for i in tqdm(range(len(data))):
            _m = list(preds.keys())[0]
            gt = preds[_m + '_gt'][i]
            gts.append(gt)
            preds_arrays = [v[i] for k, v in preds.items() if "_gt" not in k]
            tmp_preds_array = []
            for crit in range(len(preds_arrays[0])):
                crit_array = [arr[crit] for arr in preds_arrays]
                lab = np.random.rand(1)[0]
                tmp_preds_array.append(
                    1 if sum(crit_array) >= len(preds_arrays) - t else 0
                )
            #if sum(preds_arrays) > 3: # This works for Cornish
            #    label = 1
            #else:
            #    label = 0
            #    #if np.random.rand(1)[0] > 0.7:
            #    #    label = 1
            #if preds_arrays[1] == 0: # This breaks AAVE
            #    label = 0
            if sum(tmp_preds_array) == len(['c1', 'c2a', 'c2b', 'c3', 'c4', 'c5']) - t: 
                label = 1
            else:
                label = 0
                #if np.random.rand(1)[0] > 0.7:
                #    label = 1
            preds_list.append(label)
        after_acc = round(np.average([p == q for p, q in zip(gts, preds_list)])*100, 1)
        after_f1 = round(f1_score(gts, preds_list, average="weighted")*100, 1)
        print("Majority vote", t, after_acc, after_f1)
    ##print(mcnemar_test(preds_finetuned, preds_list, gts)[0])


In [ ]:
boost("West_Frisian")

0 acc 75.0 f1 74.5
1 acc 68.0 f1 66.1
2 acc 63.7 f1 59.6
3 acc 59.3 f1 52.4
4 acc 56.0 f1 46.5


In [ ]:
boost("AAVE")

0 acc 80.2 f1 83.8


100%|██████████| 1013/1013 [00:00<00:00, 37776.78it/s]


Majority vote 0 53.4 63.1


100%|██████████| 1013/1013 [00:00<00:00, 43169.95it/s]


Majority vote 1 16.6 19.4


100%|██████████| 1013/1013 [00:00<00:00, 55198.97it/s]

Majority vote 2 8.3 3.2


In [ ]:
boost("Geordie")

0 acc 56.9 f1 54.0


100%|██████████| 1012/1012 [00:00<00:00, 66636.87it/s]


Majority vote 0 47.3 43.3


100%|██████████| 1012/1012 [00:00<00:00, 69989.21it/s]


Majority vote 1 42.9 36.2


100%|██████████| 1012/1012 [00:00<00:00, 68810.36it/s]

Majority vote 2 38.7 24.8


In [ ]:
boost("Cornish")

0 acc 56.4 f1 50.5


100%|██████████| 1012/1012 [00:00<00:00, 76917.87it/s]


Majority vote 0 57.7 42.3


100%|██████████| 1012/1012 [00:00<00:00, 68183.63it/s]


Majority vote 1 59.2 59.4


100%|██████████| 1012/1012 [00:00<00:00, 74915.47it/s]

Majority vote 2 55.6 43.9


In [ ]:
boost("Yorkshire")

0 acc 77.8 f1 84.7


100%|██████████| 1007/1007 [00:00<00:00, 53042.45it/s]


Majority vote 0 47.5 61.2


100%|██████████| 1007/1007 [00:00<00:00, 70958.52it/s]


Majority vote 1 13.8 19.5


100%|██████████| 1007/1007 [00:00<00:00, 66682.41it/s]

Majority vote 2 5.0 4.1
